# 02 — Exploratory Data Analysis
### DiscoverAI · Deloitte × LUISS

EDA on the final cleaned dataset produced by notebook 01.
All numbers here reflect the state **after** cleaning v4.

**Sections:**
1. Dataset overview — coverage, token distribution, reviews per product
2. Rating distribution — products and reviews, 5-star bias before/after
3. Text quality — token counts, helpful_vote distribution, v1 vs v3 comparison
4. Price distribution — price_bucket breakdown
5. Quality score & dynamic alpha — embedding quality signals
6. Summaries & entities — BART coverage, top extracted ingredients
7. Final summary printout


## 0 · Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["font.family"] = "sans-serif"

DATA_DIR = "/Users/danielegiovanardi/MSTERRORS/Mean-Squared-Terrors/data"  # <- change this

prod = pd.read_csv(os.path.join(DATA_DIR, "products_cleaned.csv"))
rev  = pd.read_csv(os.path.join(DATA_DIR, "reviews_cleaned.csv"))
prof = pd.read_csv(os.path.join(DATA_DIR, "product_profiles.csv"))

print(f"Products: {len(prod):,}")
print(f"Reviews : {len(rev):,}")
print(f"Profiles: {len(prof):,}")


## 1 · Dataset overview

Three charts:
- **Left:** field coverage — which fields are populated and how often
- **Center:** token distribution in `product_text_base` (mean vs median)
- **Right:** reviews per product distribution


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Overview — dataset finale v3", fontweight="bold", fontsize=13)

# Copertura campi prodotto
fields  = ["brand","features_clean","description_clean","details_text","price"]
labels  = ["Brand","Features","Description","Details/Specs","Price"]
pcts    = [prod[f].notna().mean()*100 for f in fields]
colors  = ["#1D9E75" if p > 60 else "#BA7517" if p > 25 else "#D85A30" for p in pcts]
bars = axes[0].barh(labels, pcts, color=colors, edgecolor="white")
axes[0].axvline(100, color="grey", ls="--", lw=0.8, alpha=0.5)
axes[0].set(title="Product field coverage (%)", xlabel="%", xlim=(0, 110))
for bar, pct in zip(bars, pcts):
    axes[0].text(pct + 1, bar.get_y() + bar.get_height()/2,
                 f"{pct:.0f}%", va="center", fontsize=10)

# Distribuzione token product_text_base
axes[1].hist(prod["text_tokens"].clip(upper=300), bins=50,
             color="#7B68EE", edgecolor="white", linewidth=0.3)
axes[1].axvline(prod["text_tokens"].mean(),   color="red",    ls="--", lw=1.5,
                label=f"mean={prod['text_tokens'].mean():.0f}")
axes[1].axvline(prod["text_tokens"].median(), color="orange", ls="--", lw=1.5,
                label=f"median={prod['text_tokens'].median():.0f}")
axes[1].set(title="Tokens per product (product_text_base)", xlabel="Token", ylabel="# Prodotti")
axes[1].legend(fontsize=9)

# Review per prodotto
rpp = rev.groupby("parent_asin").size()
axes[2].hist(rpp.clip(upper=16), bins=14, color="#185FA5", edgecolor="white", linewidth=0.3)
axes[2].axvline(rpp.mean(), color="red", ls="--", lw=1.5, label=f"mean={rpp.mean():.1f}")
axes[2].set(title="Reviews per product", xlabel="# Review", ylabel="# Prodotti")
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Total products:          {len(prod):,}")
print(f"Total reviews:            {len(rev):,}")
print(f"Reviews/product (mean):  {len(rev)/len(prod):.1f}")
print(f"Mean tokens per product:      {prod['text_tokens'].mean():.0f}")
print(f"Products < 20 tokens:      {(prod['text_tokens']<20).mean():.1%}  (poor text)")
print(f"Products > 100 tokens:     {(prod['text_tokens']>100).mean():.1%}  (rich text)")


## 2 · Rating distribution

Three charts:
- **Left:** product average rating distribution
- **Center:** review rating distribution after balanced selection
- **Right:** 5-star bias comparison — raw vs v1 (top-10 helpful) vs v3 (balanced)

The third chart is the key result: we reduced 5-star bias from 70% to 48%.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Rating distribution — products and reviews", fontweight="bold", fontsize=13)

# Rating prodotti
axes[0].hist(prod["product_avg_rating"].dropna(), bins=30,
             color="#1D9E75", edgecolor="white", linewidth=0.3)
axes[0].axvline(prod["product_avg_rating"].mean(), color="red", ls="--", lw=1.5,
                label=f"mean={prod['product_avg_rating'].mean():.2f}")
axes[0].set(title="Product average rating (Amazon)", xlabel="Rating", ylabel="# Prodotti")
axes[0].legend()

# Rating review — distribuzione bilanciata
r_counts = [rev[rev["rating"]==r].shape[0] for r in [1,2,3,4,5]]
r_colors = ["#E24B4A","#F09595","#FAC775","#97C459","#1D9E75"]
bars = axes[1].bar([1,2,3,4,5], r_counts, color=r_colors, edgecolor="white")
axes[1].set(title="Review rating distribution (post-cleaning v4)", xlabel="Stelle", ylabel="# Review")
total_r = sum(r_counts)
for bar, cnt in zip(bars, r_counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f"{cnt/total_r:.0%}", ha="center", fontsize=10)

# Confronto bias 5-star: raw vs cleaning v3
versions = ["Raw\n(494k)", "v1 Top-10\nhelpful", "v3 Balanced\n(73k)"]
pct_5star = [61.1, 70.0, 47.9]
colors_v  = ["#D85A30", "#BA7517", "#1D9E75"]
bars2 = axes[2].bar(versions, pct_5star, color=colors_v, edgecolor="white", width=0.5)
axes[2].axhline(50, color="grey", ls="--", lw=1, alpha=0.7, label="50%")
axes[2].set(title="5-star bias: raw vs v1 vs v4", ylabel="% review 5-star", ylim=(0, 80))
for bar, pct in zip(bars2, pct_5star):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f"{pct:.1f}%", ha="center", fontweight="bold", fontsize=11)
axes[2].legend()

plt.tight_layout()
plt.show()

print("Review rating distribution:")
for r in [1,2,3,4,5]:
    n = (rev["rating"]==r).sum()
    print(f"  {r}⭐: {n:,} ({n/len(rev):.1%})")


## 3 · Text quality

Three charts:
- **Left:** review token distribution with 25-token threshold marker
- **Center:** mean tokens per product across versions (v1 → v3 → v4)
- **Right:** helpful_vote distribution (pie chart)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Text quality — products and reviews", fontweight="bold", fontsize=13)

# Token review
axes[0].hist(rev["tok_len"].clip(upper=200), bins=50,
             color="#185FA5", edgecolor="white", linewidth=0.3)
axes[0].axvline(rev["tok_len"].mean(),   color="red",    ls="--", lw=1.5,
                label=f"mean={rev['tok_len'].mean():.0f}")
axes[0].axvline(rev["tok_len"].median(), color="orange", ls="--", lw=1.5,
                label=f"median={rev['tok_len'].median():.0f}")
axes[0].axvline(25, color="green", ls=":", lw=2, label="soglia min=25")
axes[0].set(title="Tokens per review (after ≥25 filter)", xlabel="Token", ylabel="# Review")
axes[0].legend(fontsize=8)

# Confronto token prodotto v1 vs v3
cat   = ["v1\n(media 27)", "v3 prima\nenrichment\n(media 72)", "v3 dopo\nenrichment\n(media 85)"]
vals  = [27, 72, 85]
c_tok = ["#D85A30", "#BA7517", "#1D9E75"]
bars3 = axes[1].bar(cat, vals, color=c_tok, edgecolor="white", width=0.5)
axes[1].set(title="Mean tokens in product_text_base: v1 vs v4", ylabel="Token medi")
for bar, v in zip(bars3, vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(v), ha="center", fontweight="bold", fontsize=13)

# Copertura helpful_vote
hv_buckets = ["0 voti", "1-4 voti", "5-19 voti", "20+ voti"]
hv_counts  = [
    (rev["helpful_vote"]==0).sum(),
    ((rev["helpful_vote"]>=1)&(rev["helpful_vote"]<5)).sum(),
    ((rev["helpful_vote"]>=5)&(rev["helpful_vote"]<20)).sum(),
    (rev["helpful_vote"]>=20).sum(),
]
hv_colors = ["#D85A30","#FAC775","#97C459","#1D9E75"]
wedges, texts, autotexts = axes[2].pie(
    hv_counts, labels=hv_buckets, colors=hv_colors,
    autopct="%1.1f%%", startangle=90,
    textprops={"fontsize": 9}
)
axes[2].set_title("helpful_vote distribution in reviews")

plt.tight_layout()
plt.show()

print(f"Mean review tokens: {rev['tok_len'].mean():.0f}")
print(f"Median review tokens:     {rev['tok_len'].median():.0f}")
print(f"P90 review tokens:               {rev['tok_len'].quantile(0.9):.0f}")
print(f"With helpful_vote > 0:   {(rev['helpful_vote']>0).mean():.1%}")


## 4 · Price distribution

Price data is only available for 22.5% of products. This is a known limitation
of the dataset — we handle it by making price filters soft (they only apply
when the bucket field is populated).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Price distribution", fontweight="bold", fontsize=13)

# Price bucket
bucket_order  = ["budget","low","mid","high","premium"]
bucket_labels = ["Budget\n(<$10)","Low\n($10-25)","Mid\n($25-50)","High\n($50-100)","Premium\n(>$100)"]
bucket_counts = [prof[prof["price_bucket"]==b].shape[0] for b in bucket_order]
b_colors      = ["#1D9E75","#5DCAA5","#FAC775","#F09595","#E24B4A"]
bars4 = axes[0].bar(bucket_labels, bucket_counts, color=b_colors, edgecolor="white")
axes[0].set(title=f"Price bucket distribution (only 22.5% have price)", ylabel="# Prodotti")
for bar, cnt in zip(bars4, bucket_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f"{cnt:,}", ha="center", fontsize=10)

# Distribuzione prezzi disponibili (escluso outlier)
prices = prof["price"].dropna()
prices_clip = prices[prices < 150]
axes[1].hist(prices_clip, bins=40, color="#7B68EE", edgecolor="white", linewidth=0.3)
axes[1].axvline(prices.median(), color="red", ls="--", lw=1.5,
                label=f"mediana=${prices.median():.2f}")
axes[1].axvline(prices.mean(),   color="orange", ls="--", lw=1.5,
                label=f"media=${prices.mean():.2f}")
axes[1].set(title=f"Distribuzione prezzi disponibili (n={len(prices_clip):,}, clip $150)",
            xlabel="Prezzo ($)", ylabel="# Prodotti")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Products with price: {prof['price'].notna().sum():,} ({prof['price'].notna().mean():.1%})")
print(f"Median price:  ${prices.median():.2f}")
print(f"Mean price:    ${prices.mean():.2f}")
print(f"Max price:      ${prices.max():.2f}")
print(f"\nPrice bucket breakdown:")
print(prof["price_bucket"].value_counts().to_string())


## 5 · Quality score and dynamic alpha

**Quality score** combines three signals:
- Product rating (40%) — normalised to [0,1]
- Sentiment from reviews (35%) — pct_positive − 0.5 × pct_negative
- Helpful credibility (25%) — log-normalised total helpful votes

**Dynamic alpha** per product: products with weak review signals get
alpha closer to 0.70 (product embedding dominates); products with many
high-quality reviews get alpha closer to 0.35 (review embedding matters more).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Quality score and dynamic alpha per product", fontweight="bold", fontsize=13)

# Quality score
axes[0].hist(prof["quality_score"].dropna(), bins=40,
             color="#1D9E75", edgecolor="white", linewidth=0.3)
axes[0].axvline(prof["quality_score"].mean(), color="red", ls="--", lw=1.5,
                label=f"media={prof['quality_score'].mean():.3f}")
axes[0].set(title="Quality score distribution", xlabel="Score [0,1]")
axes[0].legend()

# Alpha dinamico
axes[1].hist(prof["alpha"].dropna(), bins=40,
             color="#7B68EE", edgecolor="white", linewidth=0.3)
axes[1].axvline(prof["alpha"].mean(), color="red", ls="--", lw=1.5,
                label=f"media={prof['alpha'].mean():.3f}")
axes[1].axvline(0.45, color="green", ls=":", lw=1.5, alpha=0.7, label="threshold 0.45")
axes[1].axvline(0.65, color="orange", ls=":", lw=1.5, alpha=0.7, label="threshold 0.65")
axes[1].set(title="Dynamic alpha distribution", xlabel="Alpha")
axes[1].legend(fontsize=8)

# Scatter quality vs rating
sample = prof.dropna(subset=["quality_score","product_avg_rating"]).sample(
    min(2000, len(prof)), random_state=42)
sc = axes[2].scatter(sample["product_avg_rating"], sample["quality_score"],
                     alpha=0.3, s=8, c=sample["popularity_score"],
                     cmap="YlGn")
plt.colorbar(sc, ax=axes[2], label="Popularity score")
axes[2].set(title="Quality score vs product rating",
            xlabel="Rating medio (Amazon)", ylabel="Quality score")

plt.tight_layout()
plt.show()

print(f"Quality score: mean={prof['quality_score'].mean():.3f}  "
      f"min={prof['quality_score'].min():.3f}  max={prof['quality_score'].max():.3f}")
print(f"Alpha: mean={prof['alpha'].mean():.3f}")
print(f"  < 0.45 (review embedding dominates): {(prof['alpha']<0.45).mean():.1%}")
print(f"  0.45–0.65 (balanced):                {((prof['alpha']>=0.45)&(prof['alpha']<=0.65)).mean():.1%}")
print(f"  > 0.65 (product embedding dominates):  {(prof['alpha']>0.65).mean():.1%}")


## 6 · Summary and entity coverage

Coverage of BART summaries and spaCy entity extraction.
The low ingredient/certification coverage (9-20%) reflects the fact
that most products do not have structured ingredient lists in their metadata.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Summary and entity extraction coverage", fontweight="bold", fontsize=13)

# Copertura summaries
summ_cats   = ["Full\nsummary", "With pros", "With cons", "BART\nmethod", "Extractive\nmethod"]
summ_vals   = [
    prof["summary_full"].notna().mean()*100,
    (prof["pros"].fillna("")!="").mean()*100,
    (prof["cons"].fillna("")!="").mean()*100,
    (prof["method"]=="model").mean()*100,
    (prof["method"]=="extractive").mean()*100,
]
s_colors = ["#1D9E75","#5DCAA5","#185FA5","#7B68EE","#BA7517"]
bars5 = axes[0].bar(summ_cats, summ_vals, color=s_colors, edgecolor="white")
axes[0].set(title="Summary coverage (%)", ylabel="%", ylim=(0,110))
for bar, v in zip(bars5, summ_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f"{v:.0f}%", ha="center", fontsize=10)

# Copertura entità
ent_cats = ["Ingredients", "Certifications", "Use cases"]
ent_vals = [
    (prof["ingredients"].fillna("")!="").mean()*100,
    (prof["certifications"].fillna("")!="").mean()*100,
    (prof["use_cases"].fillna("")!="").mean()*100,
]
e_colors = ["#D85A30","#BA7517","#1D9E75"]
bars6 = axes[1].bar(ent_cats, ent_vals, color=e_colors, edgecolor="white")
axes[1].set(title="Entity recognition coverage (%)", ylabel="%", ylim=(0,110))
for bar, v in zip(bars6, ent_vals):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f"{v:.0f}%", ha="center", fontsize=11)

# Ingredienti più frequenti
all_ings = []
for row in prof["ingredients"].dropna():
    all_ings.extend([x.strip() for x in str(row).split("|") if x.strip()])
top_ings = Counter(all_ings).most_common(10)
if top_ings:
    ing_labels, ing_counts = zip(*top_ings)
    axes[2].barh(list(reversed(ing_labels)), list(reversed(ing_counts)),
                 color="#D85A30", edgecolor="white")
    axes[2].set(title="Top 10 extracted ingredients", xlabel="# Prodotti")

plt.tight_layout()
plt.show()


## 7 · Final summary — all key numbers

In [ ]:
print("=" * 60)
print("  FINAL DATASET SUMMARY — DiscoverAI v4")
print("=" * 60)
print()
print("RAW INPUT")
print(f"  Raw products:              60,293")
print(f"  Raw reviews:                494,121")
print()
print("OUTPUT (post-cleaning v4)")
print(f"  Products:                  {len(prod):,}  (+66% vs v1)")
print(f"  Reviews:                    {len(rev):,}")
print(f"  Reviews per product:       {len(rev)/len(prod):.1f} media")
print(f"  Mean tokens per product:       {prod['text_tokens'].mean():.0f}  (was 27 in v1)")
print(f"  Mean tokens per review:         {rev['tok_len'].mean():.0f}  (was 35 in raw)")
print(f"  5-star bias:               47.9%  (was 70% in v1)")
print()
print("EMBEDDING QUALITY")
print(f"  Modello:                   all-mpnet-base-v2 (768 dim)")
print(f"  Alpha medio:               {prof['alpha'].mean():.3f}")
print(f"  Prodotti alpha < 0.45:     {(prof['alpha']<0.45).mean():.1%}  (review dominano)")
print(f"  Prodotti alpha > 0.65:     {(prof['alpha']>0.65).mean():.1%}  (product domina)")
print(f"  Quality score medio:       {prof['quality_score'].mean():.3f}")
print()
print("SUMMARIES & ENTITIES")
print(f"  Con summary BART:          {(prof['method']=='model').mean():.1%}  (6,579 prodotti)")
print(f"  Con summary extractive:    {(prof['method']=='extractive').mean():.1%}  (4,958 prodotti)")
print(f"  Con pros:                  {(prof['pros'].fillna('')!='').mean():.1%}")
print(f"  Con cons:                  {(prof['cons'].fillna('')!='').mean():.1%}")
print(f"  Con ingredienti:           {(prof['ingredients'].fillna('')!='').mean():.1%}")
print(f"  Con certificazioni:        {(prof['certifications'].fillna('')!='').mean():.1%}")
print()
print("SEARCH PERFORMANCE (re-ranking test on 10 queries)")
print(f"  Delta rating medio top-5:  +0.21 stelle")
print(f"  Query migliorate:          9/10")
print(f"  Electric toothbrush:       3.58 → 4.06  (+0.48)")
print(f"  Blood pressure monitor:    4.04 → 4.42  (+0.38)")
print("=" * 60)
